# Глава 8. Метрики
В данной главе рассмотрим основные метрики, используюшиеся в текстовой аналитике. Начнем с метрик перевода. Закончим метриками языковых моделей и поговорим про бенчмарки, их типизацию и рассмотрим примеры.

## Мотивация
Допустим, мы разработали модель A. Как мы понимаем, что модель A работает лучше модели B? И, что важнее, что значит "лучше"? У арифметической задачи `2+2=?` есть только один правильный ответ (возможно, с точностью форматирования) и проверить корректность легко, мы просто считаем долю правильных ответов на размеченой тестовой выборке.

Но что делать, проверять генеративную модель? Здесь гораздо больше аспектов. Например, промпт "Напиши смешную историю", "Напиши HTTP-сервер".
в случае с генерацией критерий правильности агентными системами гораздо более многогранные

Вопрос приницпиальный не только для награждения победитиелей, это задает направление всему развитию области: модели оптимизируют то, что мы умеем измерять

В ML метрики:
- оптимизируемая
- отслеживать прогресс во время обучения: здесь нужна дешёвая, автоматическая, чувствительная метрика, которую можно считать на каждом шаге
- сравнивать готовые модели по их способностям: здесь важна не дешевизна, а то, насколько метрика отражает реальную полезность

Хочется чтобы метрика была
- универсальной
- устойчивой
- просто считаемой
- дифференцируемой, чтобы какой вклад вносит каждый параметр и могли запустить процесс оптимизации
- выпуклой

## Закон Гудхарта
В 1975 году Чарльз Гудхарт, британский экономист, сформулировал принцип: `"Как только метрика становится целью, она перестаёт быть хорошей метрикой"`. Смысл в том, что любая метрика - это прокси к интересующей нас характеристике, на нее попмимо целевого факторов влияет множество второстепеныых и часто проще оптимизировать один из них, "хакнуть систему". Он сформулировал свой принцип относительно денежной политики, но принцип применим в любой сфере

Предположим, понимание студентом предмета $V$ оценивается оценкой за экзамен $U$. Тогда оценка $U = V + \varepsilon$ складывается из понимания + случайный шум: везение с билетом, натаскивание, списывание. Но дело в том, что вторую часть оптимизировать проще. Когда все понимают, как оценка зависит от $\varepsilon$, он перестаёт быть случайным. Его наращивают нарочно, например, натаскивают себя под формат теста, пишут шпаргалки и т.д. Важно, что понимание от этого никак не улучшается

Пример самого Гудхарта: центробанк отслеживал агрегат M3 как индикатор инфляции. Как только его сделали таргетом, финансовые институты перестроили инструменты так, что деньги перетекли в формы, не попадающие в M3, — индикатор перестал показывать то, что показывал. Есть таргетировать выпуск гвоздей в штуках, их выпускают мелкими, если в тоннах — выпускают мало, но огромных. Пара других примеров. Индекс Хирша и число публикаций как мера научной продуктивности породили дробление результата на минимальные публикуемые единицы, взаимное цитирование. Кол-во написанных строк кода как мера качества работы программиста. Число закрытых тикетов стимулирует дробление задач

Что можно сделать чтобы избежать "накрутки":
- использовать несколько метрик сразу, чтобы их было труднее взломать одновременно; 
- периодически менять метрики; 
- держать часть оценки закрытой (held-out), недоступной для оптимизации; 
- сочетать количественные показатели с качественным суждением; 
- различать метрики для мониторинга и метрики для управления — первые ломаются реже, потому что на них никто не давит

## Энтропия 
Рассмотрим языковую модель.
Не стоит путать с кросс-энтропией. Считаем метрику, но считаем но относительно себя а относительно конкретного использованного токена. 

## Перпелския

Языковая модель прогнозирует вероятность следующего токена. Естественная мера качества такой модели — насколько высокую вероятность она присваивает реальному тексту, которого раньше не видела. Если модель хорошо понимает язык, настоящие тексты должны быть для неё «ожидаемыми».

Формально это выражается через кросс-энтропию — среднее отрицательное лог-правдоподобие токенов на отложенной выборке:

$$H = -(1/N) \sum \log P(x_i | x_1, ..., x_{i-1})$$

Не стоит путать кросс-энтропию с энтропией генерации. Насколько модель может воспроизводить реальные тексты, чем больше, тем лучше. Энтропия генерации - сколько вариантов продолжения есть у модели на каждом шаге. В большинстве моделей регулируется температурой $\tau$ генерации

Перплексия — это просто экспонента от кросс-энтропии:

$$
PPL = e^{H}
$$

Мы рассматривали перплексию в первой главе

Интуиция следующая: это «эффективное число равновероятных вариантов», между которыми модель в среднем колеблется на каждом шаге. Перплексия 1 означала бы идеальное предсказание (модель всегда уверена в правильном токене), а перплексия, равная размеру словаря, — полное незнание языка (модель угадывает наугад). Чем ниже перплексия, тем лучше. Историческими ориентирами служили, например, наборы вроде Penn Treebank и WikiText, на которых десятилетиями мерили прогресс языкового моделирования.

У перплексии есть важная техническая оговорка. Она зависит от токенизации и словаря, поэтому перплексии двух моделей с разными токенизаторами напрямую несравнимы. Чтобы обойти это, используют метрики, нормированные на символы или байты — bits-per-character и bits-per-byte, — которые не зависят от того, как именно текст разбит на токены, и позволяют честно сравнивать разные архитектуры.

Сильные стороны перплексии — дешевизна и отсутствие необходимости в разметке: её можно считать на любом корпусе и строить по ней кривые обучения. Именно поэтому она остаётся главной метрикой на этапе предобучения. Слабость в том, что перплексия измеряет правдоподобие и беглость, но не полезность. Модель может прекрасно предсказывать токены и при этом плохо следовать инструкциям, врать или быть бесполезной в диалоге. Поэтому с переходом к инструктивным и диалоговым моделям перплексия перестала быть достаточной, и центр тяжести сместился к внешним, поведенческим метрикам.

## Метрики перевода
Задача машинного перевода была одной из первых практических задач в NLP. Первые идеи были сформулированы еще в 1949 году. В 1954 году в ходе Джорджтаунского эксперимента IBM перевели 49 русских предложений на английский язык, использовался словарь в 250 слов и 6 правилами. Пракическая демонстрация возможностей вызвала волну интереса и финансирования. Однако к 1966 году машинный перевод признали медленнее и дороже человеческого и область затормозилась почти на 20 лет. Статистического модели языка развитие возобновилось. Эксперты начали называть задачу решенной толькок к концу 2010-х.

У МП есть несколько похожих задач, можно рассматривать как seq2seq генерацию. Близкая задача - это задача аннотирования / суммаризации. Входной и выходной языки здесь не отличаются, нужно длинную последовательность перевести в короткую с максимальным сохранением смысла 

<img1 src="img/metrics.png" width=500>

До появления автоматизированных метрик перевод преимущественно оценивалася экспертно (по шкале 1-5). С ростом кол-ва моделей процесс оценивания сделали автоматизированым. Главное, чтобы был эталонный ответ (референс), размеченный профессиональным переводчиком

Общая задача генерации формально подходит, но сложно предложить эталонный ответ.

Какие сложности: перевод почти никогда не отображет слова 1-к-1. Порядок слов может быть разный.

На чем может быть основана метрика:
- пересечение
- сопоставление
- расстояние
    - в смысле кол-ва исправлений, редакторское
    - непрервыное
- семантика
    - табличная (WordNet)
    - непрерывная (эмбединги)

Исторически компания IBM довольно много занималась переводом 

Начнем с самых простых метрик, основанных на подсчете пересечения перевода с эталоном. Они опираются на точность (precision) и полноту (recall) перевода:

$$P=\frac{|T| \cap |R|}{|T|} \quad R = \frac{|T| \cap |R|}{|R|} \text{, где T - множество токенов перевода, R - множество токенов эталона}$$ 

### BLEU
В 2001 году исследовательская группа из IBM ([Papineni et al.](https://aclanthology.org/P02-1040/)) предложили для оценки качества перевода использовать метрику, в рамках которой оба текста (перевод-кандидат и эталонный перевод) разбивались на слова (токены) и считалось пересечение этих двух множеств. Чтобы перевод на текстах разной длины был сравним, пересечение считалось не в абослютных занчениях, а как доля от общего количества слов во всем тексте. Отвечая на вопрос, относительно какого именно текста: кандадата или эталона, авторы выбрали перевод-кандидат, обосновав это тем, что в машинном переводе точность (Precision) важнее полноты (Recall). Действительно, перевести можно по-разному, главное чтобы корректные. По этой причине метрику называют метрикой, ориентированной на точность, поскольку в центре расчета находится вычисление классификационной точности, а сама метрика отвечает на вопрос "Какая доля слов перевода корректна?". Позже в этой главе увидим примеры метрик, оринетированных на полноту.

Метрика получила название **BLEU** (bilingual estimted understudy) и на десятилетие стала стандартом для оценки качества машинного перевода. До этого качество перевода оценивалось в основном через привлечение процессиональных переводчиков, что очевидно дорого и крайне немасштабируемо. Появвившаяся метрика в некотором смысле выполняла роль помощника-дублера, отсюда "understudy" в названии. Обозначим за T множество слов перевода-кандадата, а за R множество слов эталонного перевода, тогда точность перевода считается как:

$$P = \frac{|T \cap R|}{|T|}$$

В переводе особую роль играет использование выражений и словосочетаний, поэтому пересечение считают не только по отдельным словам, но и по n-граммам (обычно берут n от 1 до 4). Пересечение по униграммам оценивает выбор лексики, а по n-граммам правильность расстановки слов по нужному порядку. Получив четыре точности $P_1..P_4$, их надо свернуть в одно значение. Авторы берут от них геометрическое среднее:
$\prod_{n=1}^{4} P_n^{w_n}$. Геометрическое, поскольку оно более чувстивтельное. Если арифметическое среднее может, например, скомпенсировать провал на уровне биграммов большим значением на уровне униграммов, то геометрическое в этом случае просто уронит метрику. На практике произведение часто переписывают через сумму логарифмов. Это инженерное удобство, так меньше риск переполнения снизу при перемножении маленьких дробей:
$$\prod_{n=1}^{N} P_n^{w_n} = \exp\left(\sum_{n=1}^{N} w_n \log P_n\right)$$

Если при подсчете пересечения не требовать уникальность слов, то модель может хакнуть метрику, просто бесконечно дублируя слово, в котором она уверена (например, перевод «кот кот кот кот» при эталоне «кот сидит на коврике» дает точность 100%, ведь все слова перевода корректные). Поэтому вклад повторяющихся токенов ограничивают сверху числом его вхождений в эталон, этот прием называется clipping.

Любопытно, что метрика была задумана как "корпусная", то есть перед вычислением дроби мы сначала считаем суммарное пересечение по всем примерам тестовой выборки $\sum_{C \in \text{Candidates}}$, а только потом делим числитель на знаменатель. Обратите внимание, что это не то же самое, что считать по каждому предложению и затем усреднять, ассоциативность тут не работает. Так сделано, чтобы получить более усойчивые оценки, и короткие предложения не получали завышенного веса. В теории машинного обучения такой способ усреднения называется *микроусреднением* в противовес *макроусреднению*.

$$P^{micro} = \frac{\sum_{c} |T \cap R|}{\sum_{c} |R|} \, , \quad P^{macro} = \frac{1}{c} \sum_{|c|} \frac{|T \cap R|}{|R|}$$

Ещё один момент, который важно было учесть - метрика никак не проверяет длину перевода. Поскольку она ориентирована на точность, у модели возниакет соблазн вставлять в перевод только те слова, в которых она уверена. Это вообще может быть всего одно слово и такой перевод получит 100% точность, ведь все слова перевода корректны. Поэтому к метрике обязательно добавляют штрафной мультипликатор (Brevity Penalty) в виде убывающей экспоненты, который активизируется, если перевод-кандидат короче эталонного перевода. За слишком длинные переводы отдельного штрафа нет, но он тут и не обязателен, поскольку в этом случае точность физически не может дойти до 100%, и сама начинает падать с ростом длины перевода. Действительно, ведь длину перевода можно увеличивать только за счет добавления некорректных слов, все корректные в эталоне. Итого метрика BLEU считается как:

$$\text{BLEU} = \exp\left(\sum_{n=1}^{N} w_n \log p_n\right) \cdot \begin{cases} 1 & \text{если } c > r \\ e^{(1 - r/c)} & \text{если } c \le r \end{cases}$$

Важно отметить, что значение метрики зависит от препроцессинга: выбранной токенизации, регистра текста, обработки пунктуации, поэтому сравнивать разные модели нужно очень аккуратно. Чтобы упростить расчет на практике, существует библиотека __SacreBLEU__, предоставляющая стандартную реализацию метрики BLEU. Существует разновидность метрики от Google, которая называется __GLEU__, и она считается как минимум из точности и полноты по n-граммам. Это призвано сгладить перекос оригинальной метрики в сторону точности.

### ROUGE
В отличие от задачи машинного перевода задачу суммаризацию логичнее оценивать не по точности, а по полноте (recall). Действительно, не так критично, если в суммаризацию попадут какие-то лишние слова, которых нет в эталоне. Куда важнее, чтобы сумаризация содержала необходимый минимум ключевых слов из оргинала.

В 2004 году [(Chin-Yew Lin et al)](asfafs) взяли метрику BLEU и модифицировали так, чтобы вместо точности она опиралась на полноту. Метрику назвали **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) чтобы подчеркнуть её родство с BLEU (с французского переводится, соотвественно, "красный" и "синий"). В самой формуле разница минимальная, в знаменателе количество слов в переводе-кандидате заменили на количество слов в эталонном переводе. Теперь метрика отвечает на вопрос "Какая доля слов эталона попала в перевод?".

$$\text{ROUGE(N)} = \frac{|T \cap R|}{|R|}$$

Но новизна была скорее не в изменении формулы, а в том что, авторы попытались описать целое семейство формул, объединенных универсальной логикой. Например, если оценивается совпадение только униграмм, то это метрика ROUGE-1. Если биграмм, то ROUGE-2. Если ослабить требование и отслеживать не пары слов идущих подряд, а пары слов расположенных в одинаковом порядке, с любым количеством слов между ними (это назвали skip-биграммы), то такой вариант метрики назвали __ROUGE-S__. Если допускаем не любое кол-во слов между биграммой, а не более $k$, то это метрика __ROUGE-S$k$__. ROUGE-SU дополнительно добавляет униграммы, чтобы кандидат с одним верным словом не получал нулевой оценки. Более универсальный вариант __ROUGE-LCS__ оценивает кол-во совпадающих слов с учетом порядка (длина наибольшей общей подпоследовательность). Модификация __ROUGE-Lsum_- считает LCS отдельно по каждому предложению референса и суммируется, чтобы порядок предложений не влиял на результат.

Суммаризатор, скопировавший весь документ, получит Recall = 1. Поэтому на практике заменяют на F-меру __ROUGE-F1__:
$$F_\beta = \frac{(1+\beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R}$$

На практике часто бывает несколько эталонов, тогда берётся максимумальное значение ROUGE по всем эталонам.

### METEOR
За простоту метрики BLEU приходится платить: она не учитывает возможную синонимчность слов, игнорирует полноту ответа и вообще плохо коррелирует с экспертной оценкой. В 2005 году [(Banerjee и Lavie)](https://aclanthology.org/W05-0909/) предложили усовершенствовать метрику и назвали свою версию **METEOR** (Metric for Evaluation of Translation with Explicit ORdering). 

Во-первых, авторы провели исследование и показали, что оценки качества перевода, сделанные экспертами, ощутимо лучше коррелируют именно с полнотой, а не с точностью. Более поздние работы это неоднократно подтверждали. Поэтому, выбирая между точностью и полнотой, в качестве итоговой метрики взяли гармоническое среднее с сильным перекосом в сторону полноты (например, $\alpha = 0.9$). В такой постановке отдельные штрафы за длину не нужны, компонент полноты сам штрафует за слишком короткий перевод, а компонент точности за слишком длинный.

$$\text{METEOR} = \frac{P \cdot R}{\alpha P + (1-\alpha) R}$$

Во-вторых, для учета синонимичности стали сравнивать сравнивать символьные конструкции разных уровней - сначала полные совпадения слов, затем основы слов (*бежал* ↔ *бежит*), синонимы по базе WordNet (*машина* ↔ *автомобиль*) и перефразировки по таблице парафраз. Если находится несколько соотвествий, то выбираем ближайшее (minimum crossings), таким образом поощряются переводы, максимально сохраняюшие порядок слов.

METEOR также поощряет сохранение структуры. Однако делает это более гибко, через **чанки** (chunk) — максимальной группы слов, идущих подряд и в одном порядке в обоих текстах. Если перевод идеален, все слова образуют один чанк: $ch = 1$. Если слова верны, но порядок случаен, каждое слово — свой чанк: $ch = m$. Отношение $ch/m$ — степень фрагментации: чем ближе к 1, тем более рваный перевод. За фрагментацию надо платить штраф:

$$\text{METEOR} = \frac{P \cdot R}{\alpha P + (1-\alpha)R} \cdot \left(1 - \gamma \left(\frac{ch}{m}\right)^{\beta}\right)$$

Стандартно $\gamma = 0.5$, $\beta = 3$.  Множитель $\gamma$ ограничивает штраф сверху 50% — даже полностью перемешанный перевод теряет не больше половины оценки

Параметры $\alpha, \beta, \gamma$ подбирают под конкретную языковую пару, максимизируя корреляцию с человеческими оценками на данных WMT. В более поздних версиях метрики добавили ещё и веса по типам совпадений (например $W_{exact}$ важнее, чем $W_{paraphrase}$ и т.д.) и дискаунтинг функциональных слов.

Метрика __RIBES__ (Rank-based Intuitive Bilingual Evaluation Score) так же основана на сопоставлении слов (alignment map), но она призвана не ограничиваться локальным порядком, а глобальным. Поэтому измеряют не пересечением, а таким показателем как ранговая корреляция Кендалла. Создана для языковых пар с сильно различающимся порядком слов (японский↔английский), где BLEU не чувствителен к перестановкам на дальних дистанциях.

[(Popovic, 2015)](https://aclanthology.org/W15-3049.pdf) предложила считать совпадения не слов, а символьных n-граммов. Такую метрику назвали __chrF__. Это один из немнгочисленных примеров символьных метрик. К сильным стронам можно отнести то, что. К слабым. 

$$\text{chrF}\beta = (1 + \beta^2) \cdot \frac{\text{chrP} \cdot \text{chrR}}{\beta^2 \cdot \text{chrP} + \text{chrR}}$$

### LEPOR
Метрика LEPOR (Length Penalty, Precision, n-gram Position difference Penalty and Recall) была предложена (Han et al) в 2012 году как независимая от языка альтернатива BLEU и METEOR. Метркиа представляет собой поизведение трех компонент:

$$\text{"гармончиеское среднее точности и полноты"}  \, \times \,   \text{"штраф за некорректную длину генерации"} \, \times \, \text{"штраф за некорректное положение слова"}$$

Штраф за длину наказывает слишком короткие или слишком длинные генерации. Здесь форма точно такая же как в метрике BLEU: $\text{LP} = \exp(1 − r/c)$.

Штраф за то, насколько далеко слово уехало от своего правильного места в эталоне
$$\text{NPosPenal} = e^{−\text{NPD}}, \quad \quad   \text{NPD} = \frac{1}{\text{Length hyp}} \times \sum |PD_i|$$

Так же как и с метрикой ROUGE, авторы предлагают сразу несколько вариантов метрики LEPOR. Если вместо произведения используется взвешенное гармоническое среднее всех трёх факторов, такой вариант называется hLEPOR. Если вместо слов, метрика пересечение n-граммов, то это nLEPOR. Если испольщуется агрегация на уровне корпуса, то это LEPOR_A, если по предложениям, то LEPOR-B.

К сильным сторонам можно отнести то, что метрика явно учитывает порядок слов. Симметричный штраф за длину. Не требует лингвистических ресурсов (в отличие от METEOR с WordNet), поэтому применима к любому языку. Настраиваемая.

К слабым сторонам то, что много гиперпараметров, которые нужно тюнить под языковую пару, иначе преимущество перед BLEU теряется; никакой семантики — синонимы и парафразы не распознаются; вычислительно тяжелее BLEU; заметно менее распространена, из-за чего результаты сложнее сравнивать с литературой. Сегодня в задачах, где важна корреляция с человеком, её обычно вытесняют нейросетевые метрики — COMET, BLEURT, BERTScore.

## Перестановочные метрики

Следующая группа метрик основана на перестановках слов.

### WER
Метрика __WER__ (Word Error Rate) основана на вычислении расстония между переводами T и эталоном R. Расстояние - какое минимальное количество элментарных модификаций нужно сделать, чтобы привести перевод к эталону:
$$\mathrm{WER} = \frac{S + D + I}{N} \quad TER = \frac{(S + D + I + \text{Shifts})}{\text{N}}$$
где $S$ — кол-во замен слова (substitutions), $D$ - кол-во удалений слов, $I$ - кол-во вставок слова, $C$ — верные слова, $N$ - длина референса

Например, $\text{WER}(\text{"sdfs"}, \text{"sdfsdf"}) = 5$

Метрика - это нормированное расстояние Левенштейна на уровне слов

Метрика __TER__ (Translation Edit Rate) - то же самое, но разрешается еще одна модификация сдвиг целого блока слов:
$$TER = \frac{(S + D + I + \text{Shifts})}{\text{N}}$$

### NIST
Годом позже появилась модификация метрики BLEU, в которой к единичный вес совпдения заменили на более информативную обратную частоту (IDF). При агрегации показателей геометрическое среднее $\sqrt{p_1 p_2 p_3 p_4}$ заменили арифметическим $p_1 p_2 p_3 p_4$. Кроме того, штраф за краткость сделали более мягким

$$\sum_{n=1}^{N}
\left\{
\frac{\displaystyle\sum_{w_1\ldots w_n \in \text{match}} \mathrm{Info}(w_1\ldots w_n)}
{\displaystyle\sum_{w_1\ldots w_n \in \text{hyp}} 1}
\right\}
\cdot \exp\!\left\{\beta \log^2
\left[\min\!\left(\frac{L_{\mathrm{hyp}}}{\bar{L}_{\mathrm{ref}}},\, 1\right)\right]\right\}$$

$$\text{где } \mathrm{Info}(w_1 \ldots w_n) =
\log_2 \frac{\mathrm{count}(w_1 \ldots w_{n-1})}{\mathrm{count}(w_1 \ldots w_n)}$$

Метрика NIST действительно коррелировала лучше с человеческими оценками, но заменить BLEU так и не смогла. Во-первых, она все-таки требовала расчета обратных частот по всему корпусу. Во-вторых шкала не интерпретируема.

## Нейросетевые метрики
С начала 2000-х модели ушли далеко вперед в плане учета семантики, а метрики все оставались синтаксическими (сравнивали на уровне строк). А что если сравнивать в семантическом пространстве?

### BERTScore 
В 2019 году один из первых вариантов такой метрики предложили [(Zhang et al., 2019)](https://arxiv.org/abs/1904.09675), которые задействовали для этого популярную на тот момент артхитектуру модели BERT, и назвали метрику __BERTScore__

Модель сопоставляет перевод с эталоном не по самим токенам, а по их контекстным эмбеддингам - обогащенным описаниям, генерируемых как выход на последнем слое Трансформера непосредственно перед генерацией конкретного токена.

Почему: выходные эмбединги содержат всю богатую семантику токена, благодаря этому улавливает синонимию и перефразирование

Обозначим машинный перевод за $\hat{x} = \langle \hat{x}_1, ..., \hat{x}_k \rangle$, эталонный перевод за $x = \langle x_1, ..., x_m \rangle$$

Для каждого токена перевода считается расстояние до каждого токена эталона и выбирается наибольшее. Не всегда полное совпадение, важно еще чтобы контекст был одинаковым.
Вместо точного совпадения токенов — мягкое соответствие. Для каждого токена берём самый похожий токен в другом тексте:

$$R_{BERT} = \frac{1}{|x|}\sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} x_i^\top \hat{x}_j \quad\quad P_{BERT} = \frac{1}{|\hat{x}|}\sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} x_i^\top \hat{x}_j \quad\quad F_{BERT} = 2\,\frac{P_{BERT} \cdot R_{BERT}}{P_{BERT} + R_{BERT}}$$

Векторы предварительно нормированы, поэтому скалярное произведение — это косинус. Структура формул та же, что в ROUGE-1: Recall — покрытие референса, Precision — оправданность кандидата. Отличие только в том, что вместо индикатора «совпало/не совпало» стоит вещественное число от 0 до 1.

Сопоставление жадное, а не оптимальное — каждый токен независимо выбирает лучшую пару. Это дешевле полноценного венгерского алгоритма и на практике работает не хуже

Опционально токены можно взвесить по обратной частоте: совпадение по слову *«и»* будет весить, чем по слову *«ратификация»*. Так смешиваются семантика и классический полход на частотности

$$R_{BERT} = \frac{\sum_{x_i \in x} idf(x_i)\max_j x_i^\top \hat{x}_j}{\sum_{x_i \in x} idf(x_i)}$$

Сырые косинусы лежат в узком диапазоне — случайные предложения дают около 0.7–0.8, и различия между системами выглядят микроскопическими. Поэтому для лучшей интерпретируемости оценку линейно растягивают относительно базового уровня $b$, посчитанного на случайных парах:
$$\hat{F} = \frac{F_{BERT} - b}{1 - b}$$



### BLEURT

Несмотря на прорыв относительно базовых метрик, метрика BERTscore изначально не обучалась на оценку качества. Она просто использует готовую языковую модель как источник представлений. Но семантическая близость не всегда равно качество перевода: модель может не знать, например, что пропуск отрицания катастрофичен, а перестановка придаточного — нет

[(Sellam et al., 2020)](sdsd) предложили, давайте  моделировать человеческую оценку напрямую. Изобретать велосипед не нужно, у нас есть трансформерная модель для работы стекстами. Так появилась метрика BLEURT

Берём BERT, подаём на вход сконкатенировать пару  (кандидат, референс) в один текст, а поверх `[CLS]`-токена вешаем один линейный слой:

$$\text{BLEURT} = W\tilde{v}_{[CLS]} + b$$

Данную модель обучаем под задачу регрессии на человеческих оценках с МНК функцией потерь

Основная проблема - человеческих оценок мало. Их десятки тысяч, они дорогие, привязаны к конкретным годам и языковым парам. Модель, обученная только на них, переобучается и не переносится на новые системы

Ключевая идея BLEURT. Между обычным BERT-предобучением и файнтюном на людях вставляется этап на миллионах синтетических пар.

Пары генерируются из предложений Википедии:
- маскирование и восстановление через BERT
- обратный перевод (back-translation)
- случайное удаление слов

Каждая пара размечается автоматически — набором дешёвых сигналов сразу:
- BLEU, ROUGE, BERTScore между парой
- вероятности обратного перевода
- entailment-метки от модели NLI (следует / противоречит / нейтрально)

Модель обучается предсказывать все эти сигналы одновременно (multi-task). Смысл: она заранее выучивает, что вообще бывает с текстом и какие искажения важны, а потом дообучается на людях уже подготовленной

### COMET
[(Rei et al., 2020)](https://arxiv.org/abs/2009.09025) продолжили ту же логику, но к переводу-кандидату и эталонному переводу добавили ещё оригинальное предложение. Метрику назвали COMET. Идея в том, что вариантов хорошего перевода много,  и не обязательно сильно завязываться на эталонный. Если сохраняется корректность относительно оригинала, источник может это показать.

<img src="img/comet.png" width=350>

Все три сегмента кодируются раздельно одним многоязычным энкодером (XLM-R), затем пулингом сворачиваются в векторы $s, h, r$. Из них собирается вектор признаков:
$$x = [h;\ r;\ h \odot s;\ h \odot r;\ |h - s|;\ |h - r|]$$

Здесь $\odot$ — поэлементное произведение, $|\cdot|$ — модуль разности. Произведение улавливает согласованность, разность — расхождение. Дальше feed-forward сеть выдаёт скаляр, обучение — регрессия на человеческих оценках (DA, MQM).

Раздельное кодирование — важное отличие от BLEURT: сегменты не видят друг друга внутри энкодера, взаимодействие вынесено в явные признаки. Это делает эмбеддинги переиспользуемыми и ускоряет вычисление

Существует также альтернативная постановка: вместо регрессии обучаем пвнжирование на триплетах — перевод VS «лучший перевод», «худший перевод», и вытягиваем их в пространстве так, чтобы лучший был ближе к источнику и референсу. Функция потерь: Triplet margin loss. Даёт более робастное ранжирование, но менее интерпретируемые абсолютные значения

Если эталонного перевода вообще нет, можно убрать $r$ из входа, такая модификация называется **COMET-QE** (quality estimation), только по паре «оригинал — перевод».

К слабым местам метрики можно отнести следующее. Во-первых, непрозрачность - если в метрике BLEU сразу поянтно, откуда взялось число, COMET гораздо менее интерпретируема. Частично решает модификация xCOMET, которая заточена под интервальную разметку. Во-вторых, несопоставимость между версиями. Значение зависит от чекпойнта: COMET 0.85 на wmt20 и на wmt22 — разные вещи. Версию модели надо репортить всегда, как SacreBLEU репортит подпись токенизации. В тртьих, предвзятость. Метрика впитывает предпочтения переводчиков и систем, на которых обучалась.

Рекомендация WMT последних лет: репортить нейросетевую метрику как основную (COMET или BLEURT) и chrF или BLEU как вспомогательную для сопоставимости с прошлыми работами. Использование только метрики BLEU для оценки современных систем уже считается недостаточным.

### LLM-as-a-judge
Наиболее универсальный вариант — подход __LLM-as-a-judge__, который к середине 2020-х стал доминирующим способом оценки открытой генерации. Берется какая-то мощная языковая модель (из топа), получает запрос, один или несоклько сгенерированных ответов для оценку + инструкцию по оцениванию, например, можно попросить выставить каждому решению свой score бал или просто выбрать лучший вариант.

У такого подхода к оценке появляется свойство масштабирумости - можно разметить миллионы генераций, не тратя много денег и времени. Качество при этом будет коррелировать с человеческой разметкой.

У модели оценщика однако тоже могут быть свои систематические искажения, о которых важно знать: 
- предпочтение собственного стиля и собственных ответов
- смещение в сторону более длинных и уверенно звучащих ответов
- чувствительность к порядку предъявления вариантов

Поэтому LLM-as_Judge важно калибровать. Как минимум перемешивать порядок вариантов и перепроверять часть примеров оценщиками

### Win Rate
Для сравнительных оценок ("какой из двух ответов лучше A или B?") — долю побед (__win rate__) одной модели над другой

Когда предполагается, что у задачи есть объективно проверяемый ответ, такие метрики наиболее надёжные. Сейчас вектор развития ИИ сместился в прикладную область: важно оценивать, не просто насоклько хорошо модель генерирует текст, а как она решает конкретные прикладные задачи (код, математика, агентные сценарии). В этих задачах, как правило, корректность можно проверить автоматически и однозначно

### Метрики читаемости
Иногда важно оценить "сложность" генерируемого текста. Самое простое, что можно предложить, это посчитать среднее кол-во слов в одном предложении. А также кол-во слогов или букв в одном слове. Большие значения будут говорить об испольщзовании богатого словарного запаса и наличии специфичной лексики. Для лучшей интерпретируемости эти показатели иногда отображают в класс школы, которому соотвествует словарное разнообразие текста. Например, этот ответ соотвествует лексике пятиклассника, а этот на уровне выпускника.

Либо можно посчитать долю слов с большим (3+) количеством слогов. Другой подход - оценить долю слов из топ-3000 наиболее популярных слов языка. Если ответ изобилует именно такими словами, то текст можно считать простым. При этом важно учитывать, что устная речь отличается от письменной. Кроме того, если мы пытаемся оценивать устную речь, метрики сильно зависят от того, как транскрибатор нарежет ее на предложения, могут быть ошибки. Так например, исследовали предвыборную речь Дональда Трампа и выяснили, что тот предпочитает использовать более короткие предложения по сравнению с другими кандидатами.

### Метрики разнообразия
Нужны для оценки генерируемого текста. Доля уникальных n-грамм в числе всех сгенерированных. Тут правда следует быть аккуратным, чем длинне текст, тем больше знаменатель и тем быстрее падает метрика. На генерации какой длины показатель уникальности падает ниже порога
Доля неуникальных n-грамм позволяет отлавливать зацикливание

По генеративной модели:<br>
Энтропия (не путать с кросс-энтропией на обучающем тексте), сколько вариантов продолжения текста в каждый момент времени
Сюда же можно отнести коэффициент Ципфа распределения частот токенов
MAUVE сравнивает распределения эмбедингов естественного и сгенерированного текста KL дивергенцией

Новизна - доля n-грамм, не встречающихся в обучающей выборке или промпте. Оценивает генеративные обощающие способности модели
Отдельно можно выделить метрики суммаризации: доля слов из оригинала, средняя длина экстракта, степень суммаризации. Высокие значения coeerage density говорят об экстрактивном резюме, низкие что абстрактивная

### Технические метрики
Time to first token - время от начала обработки до начала генерации prefill
Time per output token
Inter-token latency
LAtency =FT + OT

Производительность
Throughtput - кол-во запросов в единицу времении, 
Goodput - кол-во успешных запросов в единицу вермени (уложившихся в time limit)
RPS

Вычисления
FLOP
MFU  сколько возможностей железа используется
HFU включая активаций
вычислений на прочитанный байт

Использщование памяти
веса + активации + KV-кэш
контекст
квантование

стоимость запроса или обратная метрика "токенов на 1$"
Большинство провайдеров кэширует, это сильно влияет на экономику

При выводе всех метрик слежует учитывать, что 
важны персентили а не среднее
нужно указывать сопутствующую нагрузку
замерять нужно на реальном траффике


## Бенчмарки

Бенчмарк — это стандартизованный набор данных и протокол оценки, позволяющий сравнивать разные модели между собой. Ниже рассмотрим ключевые для индустрии бенчмарки.

Бенчмарк __SQuAD__, собранный в Стенфорде в 2016 году для проверки качества решения QA задач. Имеется выдержка из Википедии, нудно выделить. Рекомендуемая метрика для оценки: доля точных ответов (Exact Match) или F1 мера на пересечении слов. Бенчмарк __SNLI__ для логического следования и так далее) мерил одну узкую способность. 

В 2018 году несколько университетов собрали большой бенчмарк __GLUE__ (General Language Understanding Evaluation), ориентированный на оценку глубокого понимания текста. В нем было около 200 тысяч вопросов по разного типа, требующих так или иначе понимания текста. Всего 9 типов заданий, среди которых грамматическая корректность, определние тональности высказывания, поиск дубликатов, проверка логической связности текстов и прочее. Например, модели могут сказать, что «Мужчина идет по парку со своей собакой» и «Мужчина находится на улице» и модель должна ответить, следует ли второе утверждение из первого? По времени появление бенчмарка совпало с развитием первых Трансформерных моделей, в частности модели BERT, поэтому задачи бенчмарка очень быстро были решены и появилась потребность в более сложном тесте. Уже в 2019 году был представен усложненный вариант __SuperGLUE__, куда добавили более сложные задачи на определение причинно-следственных связей. Появиляющиеся новые модели сравнялись и превзошли человеческий результат уже в 2021 году.

Важно отметить, что ранние бенчмарки (примерно до середины 2020 года) содержали и обучающую, и тестовую часть данных, поскольку модели той эпохи еще не умели отвечать в zero-shot режиме (с впервые увиденным примером задачи в промпте), их надо было специально дообучить под задачу. После выхода модели GPT-3 перестали так делать, поскольку появилось понимание, что модели обладают достаточным общим знанием, чтобы отвечать на любые вопросы.

С развитием области, в частности с появилением рассуждающих моделей, появилась потребность в более сложных оценках. Появились бенчмарки типа __MMLU__ (Massive Multitask Language Understanding), собранного в университетской среде в 2021 году. Содержит вопросы по 57 предметам от школьного до профессионального уровня, от анатомии до юриспруденции. В 2021 OpenAI выпустил свой бенчмарк __GSM8K__, ориентированный на математику, Он содержал 8500 математических задач уровня средней школы, для решения которых требуется разбить задачу на 2-8 логических действий. Бенчмарк от Беркли __MATH__ был ориентирован на более сложные математические задачи олимпиадного уровня. Оценивали долю правильных ответов. Первые модели набирали максимум 10%. 

В том же году в OpenAI появился бенчмарк для проверки нвыков написания кода. __HumanEval__. Он содержит 164 задачи по программированию на Python. Например, ```def is_palindrome(s: str) -> bool:
    """Возвращает True, если строка читается одинаково в обоих направлениях."""``` В отличие от предыдущих тестов, где оценка производилась синтаксичесими инструментами и считала просто пересечение слов, здесь качество начали оценивать по способности проходить юнит тесты. Бенчмарк __MBPP__ (Mostly Basic Python Problems) от Google также проверял способность писать код, но был ориентирован на более бытовую постановку задач, как ее ставят реальные пользователи. Например, ```Напиши функцию на Python, которая принимает список чисел и возвращает только четные```. Содержал около 1000 задач на Python, для успеха нужно было пройти юнит-тесты.

В этот же список __HellaSwag__ от Allen Institute. Это набор вопросов, проверяющих общую адекватность размышления модели. Например, модели могут потребовать проолжить высказывание «Женщина берет кусок теста, раскатывает его скалкой на столе, берет круглую формочку и...» а) ...начинает играть на пианино б) ...вырезает круги для печенья. с) ...кладет скалку в холодильник. Оценивается, как долю правильных ответов.  ARC, WinoGrande, PIQA). Бенчмарк __TruthfulQA__, собранный в Оксфорде и проверяющий, повторяет ли модель распространённые человеческие заблуждения. 800 вопросов, описывающих типичные заблуждения. Посольку ранее было показано, что рост количества параметров не делает модели автоматически умнее, наоборот модели начинают повторять больше заблуждений.

Параллельно возникли мега-наборы и идея всесторонней оценки. __BIG-bench__ собрал более двухсот разнообразных задач, придуманных сообществом. Из него выделили особо трудное подмножество BIG-bench Hard. 

В 2022 году появился проект __HELM__ (Holistic Evaluation of Language Models) из Стенфорда, который сместил акцент с одного показателя на множество. Модель прогоняют по большому количеству сценариев и меряют не только точность, но и устойчивость, калибровку, смещения, эффективность. От «кто набрал больше» перешли в плоскость «какая модель лучше по совокупности свойств»?

К середине 2020-х область столкнулась с кризисом насыщения. Классические бенчмарки — MMLU, HellaSwag, HumanEval — топовые модели стали проходить тесты с результатом выше 90%, и различия между ними утонули в шуме. Ответом стало новое поколение более трудных оценок:

- MMLU-Pro - усложнённый MMLU с десятью вариантами ответа вместо четырёх и обязательной цепочкой рассуждений (хотя к 2026 году и он подходит к насыщению)
- GPQA-Diamond - вопросы уровня PhD по биологии, физике и химии, специально составленные так, чтобы их нельзя было нагуглить; неспециалисты набирают около трети даже с доступом в интернет
- Humanity's Last Exam — около трёх тысяч вопросов экспертного уровня от специалистов разных областей, задуманные так, чтобы оставаться трудными несколько лет

В 2019 году разработчиком из Google был создан бенчмарк __ARC-AGI__ (Abstraction and Reasoning Corpus) и позиционировался как тест на общий искусственный интеллект. Он содержит набор уникальных визуальных загадок в виде сеток с разноцветными квадратами, которые как-то трансформируются (модели показывают несколько примеров). Нужно применить такую же логику к новому рисунку. Оценивают как долю правильно решенных задач. Задачи требуют абстрактного мышления и логики. Первую версию теста научились решать к концу 2024 года, поэтому вышла вторая ARTC-AGI-2.

Бенчмарк __AIME__ (American Invitational Mathematics) содержит 15 олимпиадных задач, заимствованных изх реальных ежегодных соревнований. Требует 100 шагов рассуждений.

Отдельной и быстро растущей ветвью стали агентные бенчмарки, проверяющие не текст, а действие. Бенчмарк __SWE-bench__ (Software Engineering Benchmark) от Принстонского университета. Содержит 2300 реальных проблем (баги и пулл реквесты) из открытых репозиториев GitHub, которые нужно поичинить. Оценивается доля комитов, прошедших встроенные тесты. В 2024 году выпустили усложненный вариант __SWE-bench Verified__.

В 2023 году сообщество разработчиков вместе с Hugginface выпустили бенчмарк __GAIA__ (General AI Assistants), который содержит 500 заданий для ИИ агентов. Он испольщуется для проверки успешности действий. Это можнет быть задание типа ```Найди на сайте компании PDF-отчет с финансовыми результатами за 2024 год, найди чистую прибыль на странице 14, пересчитай ее по курсу доллара ЦБ РФ за 15 января 2025 года и назови итоговую сумму в рублях```.

Бенчмарк __WebArena__ от университета Карнеги-Мелон предоставляет среду для тестирования веб-агентов. Модели дают контроль над браузером и ставят задачу в симуляции реального интернета (интернет-магазины, форумы вроде Reddit, панели управления GitLab). Пример задачи: «Найди на форуме пользователя X, посмотри его последний пост и купи товар из его рекомендации на Amazon». Оценивают долю правильно решенных заданий.

__AgentBench__ и __tau-bench__, оценивающие работу с инструментами, навигацию и многошаговые сценарии. По мере того как модели обретают агентность, оценка тоже трансформируется от "что модель говорит" к "что модель делает". Про языковых агентов подробнее поговорим в главе 11.

### Русскоязычные бенчмарки
Для русского языка существуют свои бенчмарки.

В 2019 году Сбер выпустил __SberQuad__ русский аналог бенчмарка SQUAD, использующийся для оценки качества вопросно-ответных систем. Бенчмарк содержит порядка 50 тысяч вопросов. Вопросы такого плана: "Кто основал Москву в 1147 году?".

В 2025 году вышел бенчмарк __DRAGON__ для оценки точности динамического поиска. Чтобы модель не могла выучить правильный ответ, пул вопросов постоянно обновляется, основываясь на новостях.

В 2020 году выпустили русскую адаптацию бенчмарка GLUE __Russian SuperGlue__. Он также содержал задания 9 типов, но все вопросы были сформулированы на русском языке, суммарно около 100 тысяч вопросов. Тремя годами позже была выпущена более продвинутая версия этого датасета, которую нзавали __MERA__ (Multimodal Evaluation of Russian-language Architectures). Она уже проверяла знания модели по разным доменам (математика, биология, история). А мультимодальная версия дополнительно содержала вопросы по изображениям, аудио и видео. 

Бенчмарк __BABILong__ использовался для проверки возможностей модели удерживать внимание. Для этого модели давали прочитать объемную книгу, где в случайных позициях "разбрасывали" незаметные утверждения. Модель должна была воспроизвести все эти утверждения. 

В 2024 году выпустили __ruMTEB__ (Russian Massive Text Embedding Benchmark), являющийся русскоязычной версией бенчмарка MTEB, содержащий большой набор задач для оценки качества генерируемых эмбедингов текстов. Задачи включают классификацию текстов, кластеризацию, оценку семантического сходства, ранжирование. 





По способу выставления оценки
- автоматическая проверка по совпадению или тестам (дёшево и воспроизводимо, но применимо не везде)
- оценка моделью-судьёй (масштабируемо, но со своими искажениями)
- человеческая оценка (наиболее достоверна, но дорога и медленна)

Бенчмарки сливались в открытый доступ, их заучивали, появлявсь необзодимость постоянно придумвать новые. Предложили принципиально ругой сопосб оценивания. Вместо фиксированных задач с известными ответами он измеряет, какой ответ людям субъективно нравится больше. Самый известный пример такой среды — __LMArena__ (ранее известная как LMSYS Chatbot Arena): пользователю показывают ответы двух анонимных моделей на его собственный запрос, он выбирает лучший, а из миллионов таких попарных сравнений строится рейтинг по схеме [Elo](https://en.wikipedia.org/wiki/Elo_rating_system) или модели Брэдли–Терри. Сила арены в том, что она улавливает «ощущение полезности», которое не видят формальные бенчмарки. Слабость — в том же: люди также не лишины предвзятости: они склонны голосовать за более длинные, уверенные и красиво оформленные ответы, поэтому стиль может побеждать точность, и арена в этом случае измеряет "красоту", а не содержание

### Неравномерность внимания
С развитием моделей произошел взрывной рост максимального окна контекста: 128 тысяч, 200 тысяч, а затем и миллионы токенов. Но возможность загружать длинный текст - это одно, а насколько эффективно модель использует этот контекст - другое

Идею теста "Needle in a Haystack" предложил Грег Камрадт в 2023 году: в длинный текст вставляют одно несвязанное с темой предложение (например, утверждение, что лучшее занятие в Сан-Франциско — съесть сэндвич в парке Долорес в солнечный день) и просят модель ответить на вопрос, связанный с этим утверждением (например, "Какое лучшее занятие в Сан-Франциско?"). Эксперимент многократно повторяют, меняя локацию и общую длину контекста. Затем статистику правильных ответов рисуют тепловой картой: по одной оси — длина, по другой — глубина, цвет ячейки показывает, правльно ли ответила модель

<img src="img/needle_in_a_haystack.png" width=300>

Обнаружили что:
- способность к извлечению деградирует с ростом длины (что довольно очевидно)
- почти всегда возникает эффект __Lost in the middle__: информацию в начале и в конце контекста модели находят почти безошибочно, а вот ближе к середине провал в качестве [(Liu et al, 2023)](https://arxiv.org/abs/2307.03172)

<img src="img/lost_in_the_middle.png" width=300>

Отчасти эффект может объясняться "осторожностью" модели. Так было, например, с [ранними тестами](https://www.anthropic.com/news/claude-2-1-prompting) Claude 2.1, который  набрал низкую точность из-за того, что был обучен не отвечать на основании информации, которую считает недостаточно достоверной. Переформулировка запроса заметно меняла результа

В дальнейшем тест эволюционировал в сторону усложнения. Появились варианты с несколькими фактами (__multi-needle__), требующие интегрировать разрозненную информацию. Возник бенчмарк __RULER__ с набором синтетических длинноконтекстных задач и увидели, что большинство моделей, проходивших оригниальный тест, валятся на нем. 

В том же направлении работают __NeedleBench__, __BABILong__ (рассуждение в длинном контексте), __NoLiMa__ (поиск без буквального совпадения слов) и LongBench

### Методы борьбы с неравномерностью внимания

Существует множество методик, как бороьтся с неравномерностью внимания. Некоторые из них:

__Сортировка контекста__. Если конекст составной, то есть содержит не один документ, а набор документов, то выгоднее располагать более релевантные документы ближе к началу или концу контекста, так как там внимание более точное. Для переранжирования отобранных документов может понадобиться более сложная полносвязная модель.

__Калибровка внимания__. [(Hsieh et al, 2024)](https://arxiv.org/abs/2406.16008) предложили добавлять позиционную поправку, чтобы внимание больше веса давало токенам из середины контекста. [(Peysakhovich et al, 2023)](https://arxiv.org/abs/2310.01427) предложили перемещать ближе к краю документы, которым модель часто уделяет больше внимания. [(Su et al, 2021)](https://arxiv.org/abs/2104.09864) для уменьшения деградации на супер длинных контекстах предлагают использовать позиционные RoPE эмбединги. Мы подробно рассматривали в главе с описанием архитектуры Трансофрмеров.

__Cжатие контекста__. [(Jiang et al, 2023)](https://arxiv.org/abs/2310.05736) предложили делать дополнительное взвешивание токенов похожим на TF-IDF способом. Если перплексия токена маленькая, то мы его выкидвыаем. Другой вариант - заменять куски текста на их компактную суммаризацию, те самым упрощая работу механзиму внимания.

__Грамотное обучение__. [(An et al, 2024)](https://arxiv.org/abs/2404.16811) на этапе обучаяющих примерах располагать факт в разных локациях, не только в начале или конце. Не забывать давать примеры с инструкциями на реально длинных текстах. Это натренирует модель кстрактировать инфо в разных условиях.

__Грамотный prompt engineering__. Инструкцию дублируют в начале и конце. Решают задачу по частям, потом агрегируют (в стиле map-reduce)

__Повышение точности__. [(Lui et al, 2023)](https://arxiv.org/pdf/2407.01100) предложили на шаге Retrieval ставить выше порог релевантности, тогда будут отибраться только те документы, в полезности которых модель уверена и так контекст получается меньше. Либо доставать информацию итеративно.

## Проблемы и ограничения метрик

У бенчмарков есть слабые стороны. Давайте их разберем.

Пожалуй, главная слабое место - это утечки данных (Data contamination). Тестовые наборы публичны и со временем попадают в обучающие данные следующих моделей. Тогда высокий балл отражает не способность рассуждать, а запоминание ответов. Именно поэтому так ценятся свежие, приватные и состязательные оценки.

Вторая проблема - это быстрое насыщение. У каждого бенчмарка ограниченный срок жизни: как только модели упираются в его потолок, он перестаёт различать сильнейших, и нужен новый, более трудный. Этот эффект можно было многократно наблюдать, от GLUE до MMLU-Pro.

Уже упоминаемый в начале главы закон Гудхарта. Когда бенчмарк становится целью оптимизации, под него начинают подгонять обучение, и высокий балл может расходиться с реальной полезностью. Модель учат «сдавать экзамен», а не быть умной.

Не всегда очевидно, что бенчмарк измеряет именно ту способность, которую предполагают разработчики. Формат с выбором из вариантов, например, оставляет лазейки, которые модель может эксплуатировать, не понимая сути.

Высокий балл на академическом бенчмарке не гарантирует пользы в реальном продукте. Корреляция между лидербордами и фактическим качеством работы бывает слабой, и нередко модели с более скромными общими баллами оказываются точнее на конкретных прикладных задачах. Не говоря о субъективном восприятии польщозвателями.

Результаты чувствительны к формулировке запроса, числу примеров, версии оценочного инструментария и способу нормализации ответа, поэтому числа из разных источников не всегда сравнимы напрямую.
